<a href="https://colab.research.google.com/github/Gitstrong3333/MachineLearning_Projects2025_2026/blob/main/MedicalAssistant_FullCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## 1 - Installing and Importing Necessary Libraries and Dependencies

**Set Google Colab to use the T4 GPU**

Install `llama-cpp-python` with GPU acceleration. The wheel build is essential; ignore other errors. Then restart runtime.

- `llama-cpp-python` is a Python wrapper for llama.cpp, a universal LLM inference library that runs models efficiently using the GGUF file format.

- GGUF (GGML Universal File) is a binary format storing model weights and metadata in a single file. It uses quantization to reduce precision, decreasing memory usage and increasing inference speed.

- Model Compatibility: Supports any GGUF-converted model including Llama, Mistral, CodeLlama, Gemma, and Qwen.

- `Llama()` class: Main interface for loading and running models

- `hf_hub_download()`: A function from the Hugging Face Hub library to download specific files from Hugging Face repositories with automatic caching

In [ ]:
# Installation for GPU llama-cpp-python: Downloads and compiles the library with GPU acceleration enabled.
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

In [ ]:
!pip install -U pip setuptools wheel


In [ ]:
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 \
pip install llama-cpp-python --no-cache-dir


In [ ]:
# Install the libraries & downloading models from HF Hub
!pip install huggingface_hub pandas tiktoken==0.6.0 pymupdf==1.25.1 langchain==0.3.25 langchain-community==0.3.25 chromadb sentence-transformers numpy transformers -q

In [ ]:
# Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [ ]:
# Define the model repository and filename for the Mistral-7B-Instruct-v0.2 GGUF model.
model_repo = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_file = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [ ]:
# Download the model
model_path = hf_hub_download(
    repo_id= model_repo,
    filename= model_file
)

In [ ]:
# Initialize the model with the downloaded GGUF file.
# model_path: path to the GGUF model file.
# n_ctx: context window size (determines how much text the model can process at once).
# n_gpu_layers: number of layers to offload to the GPU for acceleration.
# n_batch: batch size for processing.
llm = Llama(
    model_path=model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


## **2.2 - Utility function generate_response**

In [ ]:
def generate_response(
    query,
    max_tokens=128,
    temperature=0,
    top_p=0.95,
    top_k=50,
    repeat_penalty=1.0
):
    """
    Generates a response from the language model.

    Args:
        query (str): The input prompt for the model.
        max_tokens (int, optional): The maximum number of tokens to generate. Defaults to 128.
        temperature (float, optional): Controls the randomness of the output. Defaults to 0.
        top_p (float, optional): Nucleus sampling parameter. Defaults to 0.95.
        top_k (int, optional): Top-k sampling parameter. Defaults to 50.
        repeat_penalty (float, optional): Penalizes repeated tokens. Defaults to 1.0.

    Returns:
        str: The generated text response.
    """
    try:
      model_output = llm(
              prompt=query,
              max_tokens=max_tokens,
              temperature=temperature,
              top_p=top_p,
              top_k=top_k,
              repeat_penalty=repeat_penalty
          )
      return model_output['choices'][0]['text'], model_output
    except Exception as e:
      return f"Error: {e}", {}

# **2.3 - Querying the LLM**
# **Query 1: What is the protocol for managing sepsis in a critical care unit?**

In [ ]:
query_1 = "What is the protocol for managing sepsis in a critical care unit?"
ans_1, moutput_1 = generate_response(query_1)
print(ans_1)
print("completion_tokens = ", moutput_1['usage']['completion_tokens'])

Llama.generate: prefix-match hit




Sepsis is a life-threatening condition that occurs when an infection triggers a cascade of inflammatory responses throughout the body. In a critical care unit, early recognition and prompt management of sepsis are crucial to improve outcomes and reduce mortality rates. Here's an outline of the protocol for managing sepsis in a critical care unit:

1. Early recognition: Recognize sepsis early by assessing patients for signs and symptoms such as fever, chills, tachycardia, hypotension, respiratory distress, altered mental status, and lactic acidosis. Use validated scoring systems like the Sequential Organ Failure Assessment (SOFA) score or the Quick Sequential Organ Failure Assessment (qSOFA) to identify patients at risk for sepsis.
2. Immediate resuscitation: Begin resuscitation efforts as soon as possible in suspected seps
completion_tokens =  200


# **Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?**

In [ ]:
query_2 = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
ans_2, moutput_2 = generate_response(query_2)
print(ans_2)
print("completion_tokens = ", moutput_2['usage']['completion_tokens'])

Llama.generate: prefix-match hit




Appendicitis is a medical condition characterized by inflammation of the appendix, a small structure that extends from the large intestine. The following are common symptoms of appendicitis:

1. Abdominal pain: The pain is usually located in the lower right side of the abdomen and may start as a dull ache but can quickly develop into a sharp, severe pain.
2. Loss of appetite: A loss of interest in food and a feeling of nausea are common symptoms of appendicitis.
3. Fever: A fever of 100.4°F (38°C) or higher is often present in individuals with appendicitis.
4. Vomiting: Nausea and vomiting may occur due to the inflammation and pressure on the stomach caused by the swollen appendix.
5. Constipation or diarrhea: Depending on the location of the
completion_tokens =  200


# **Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?**

In [ ]:
query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
ans_3, moutput_3 = generate_response(query_3)
print(ans_3)
print("completion_tokens = ", moutput_3['usage']['completion_tokens'])

Llama.generate: prefix-match hit




Sudden patchy hair loss, also known as alopecia areata, is a common condition that affects both men and women. It can cause round or oval bald patches on the scalp, but it can also occur on other parts of the body, such as the beard area or eyebrows. The exact cause of alopecia areata is not known, but it is believed to be an autoimmune condition where the immune system attacks the hair follicles.

Here are some effective treatments and solutions for addressing sudden patchy hair loss:

1. Minoxidil: Minoxidil is a topical medication that has been shown to promote hair growth in people with alopecia areata. It works by stimulating the hair follicles and prolonging the anagen phase (the growing phase) of the hair cycle. Minoxidil is available over-the-counter in liquid or foam form, and it
completion_tokens =  200


# **Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?**

In [ ]:
query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
ans_4, moutput_4 = generate_response(query_4)
print(ans_4)
print("completion_tokens = ", moutput_4['usage']['completion_tokens'])

Llama.generate: prefix-match hit




There is no one-size-fits-all answer to this question as the specific treatment recommendations for a person with a brain injury depend on the severity and location of the injury, as well as the individual's age, overall health, and other factors. However, I can provide an overview of some common treatments and therapies that may be recommended following a brain injury.

1. Medical Management: The initial focus after a brain injury is to stabilize the person's condition and manage any life-threatening conditions such as bleeding, swelling or infections. Medications may be used to control seizures, reduce intracranial pressure, manage pain, or improve cognitive function.
2. Rehabilitation: Rehabilitation is a crucial component of brain injury treatment. It typically involves a multidisciplinary team of healthcare professionals, including physicians, nurses, physical therapists, occupational therapists, speech-language pathologists,
completion_tokens =  200


# **Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?**

In [ ]:
query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
ans_5, moutput_5 = generate_response(query_5)
print(ans_5)
print("completion_tokens = ", moutput_5['usage']['completion_tokens'])

Llama.generate: prefix-match hit




First and foremost, if you suspect that someone has fractured their leg while hiking, it is essential to ensure their safety and prevent further injury. Here are some necessary precautions:

1. Keep the person calm and still: A fracture can be painful, so it's crucial to keep the person as comfortable as possible. Encourage them to take slow, deep breaths and speak softly to them to help reduce anxiety. Try to keep them still to prevent any movement that could cause further damage or discomfort.
2. Assess the injury: Check the area around the fracture for signs of swelling, bruising, or deformity. Do not attempt to realign the bone yourself as this can cause more harm than good. If you notice any open wounds, cover them with a clean dressing.
3. Provide first aid: Apply a sterile dressing to the injured area and secure it in place with a bandage or
completion_tokens =  200


# **Observation**

The generated responses are relatively generic in nature.

Additionally, the outputs are being truncated because of the default max_tokens limitation in the model configuration.

# **3 - Query LLM with Prompt Engineering and Parameter Tuning**

Prompt template for Mistral from the model card : [INST] {prompt} [/INST]

In order to leverage instruction fine-tuning, prompt is surrounded by [INST] and [/INST] tokens.

In [ ]:
# Define a simple utility function to prepare model prompt
def prepare_model_prompt(system_prompt, user_prompt):
    return f"""<s>[INST]{'system'}: {system_prompt}
                {'user'}: {user_prompt}
                [/INST]"""

# **Query 1: What is the protocol for managing sepsis in a critical care unit?**

Combination 1 - System prompt (general audience, harmless) and modified max_tokens

In [ ]:
system_prompt = """You are a helpful, respectful and honest medical assistant.
                  Always explain in simple terms for a general audience.
                  Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content.
                  Please ensure that your responses are socially unbiased and positive in nature."""

ans, moutput = generate_response(
    prepare_model_prompt(system_prompt, query_1),
    max_tokens=0,
    temperature=0,
    top_p=0.95,
    top_k=50,
    repeat_penalty=1.0
  )
print(ans)
print("completion_tokens = ", moutput['usage']['completion_tokens'])

Llama.generate: prefix-match hit


 Sepsis is a serious condition that occurs when the body has an overwhelming response to an infection. In a critical care unit, managing sepsis involves several steps to ensure the best possible outcome for the patient. Here's a simplified explanation of the protocol:

1. Recognition: Healthcare professionals must identify sepsis early and assess its severity using tools like the Sequential Organ Failure Assessment (SOFA) score or the Quick Sequential Organ Failure Assessment (qSOFA) score.

2. Resuscitation: The first priority is to stabilize the patient's vital signs, including maintaining adequate blood pressure, oxygenation, and perfusion. This may involve administering intravenous fluids, oxygen, and vasopressors.

3. Source control: Identify and address the source of the infection, such as removing an infected catheter or draining an abscess.

4. Antibiotics: Administer broad-spectrum antibiotics as soon as possible to cover the most common bacterial pathogens.

5. Supportive car

**Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?**


Combination 2 - System prompt (brevity and Shakespearean language) and modified temperature and max_tokens

In [ ]:
# temperature set to 1 and max_token is 0
system_prompt = """Respond briefly and clearly in Shakespearean language."""

ans, moutput = generate_response(
    prepare_model_prompt(system_prompt, query_2),
    max_tokens=0,
    temperature=1,
    top_p=0.95,
    top_k=50,
    repeat_penalty=1.0
  )
print(ans)
print("completion_tokens = ", moutput['usage']['completion_tokens'])

Llama.generate: prefix-match hit


 Thou askest of the common signs of appendicitis, good sir or madam? I shall oblige thee with great haste. Abdominal pain, primarily near the navel, is the first sign this malady doth present. Swelling, inflammation, and loss of appetite, as well as vomiting and a low-grade fever, are oft accompanied. Alas, good friend, no, this affliction cannot be healed by mere medicament. Instead, a surgical procedure known as appendectomy must be pursued with great haste. Forsooth, this operation, though perilous, is the only way to save the sufferer from the impending rupture and demise. Godspeed to thee, and may fortune smile upon thee in thine time of need.
completion_tokens =  175


In [ ]:
# temperature set to 1 and max_token is 0
# Repeating the same question to observe effect of temperature
system_prompt = """Respond briefly and clearly in Shakespearean language."""

ans, moutput = generate_response(
    prepare_model_prompt(system_prompt, query_2),
    max_tokens=0,
    temperature=1,
    top_p=0.95,
    top_k=50,
    repeat_penalty=1.0
  )
print(ans)
print("completion_tokens = ", moutput['usage']['completion_tokens'])

Llama.generate: prefix-match hit


 Thou askest of the telltale signs of Appendicitis, a malady most vexing? I shall oblige, fair questioner! Swelling in the lower belly, near the navel, doth oft occur. Pain thence traveleth towards right side, abdomen afflicted with great distress. Loss of appetite, and oft a feverish heat, compound this affliction. As for cure by medicine's hand, alas, it doth not oft accord. Thus, surgical procedure, in form of an appendectomy, doth become the remedy's decree.
completion_tokens =  133


Observation

The explanation style is somewhat poetic rather than strictly technical.

Additionally, when the same question is asked multiple times, the responses vary because the temperature parameter is set to 1, introducing randomness into the model’s output generation.

# **Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?**
Combination 3 - System prompt (empty) and modified top_k

top_k controls the maximum number of most-likely next tokens to consider when generating the response at each step.

In [ ]:
# top_k set to 5
system_prompt = ""

ans, moutput = generate_response(
    prepare_model_prompt(system_prompt, query_3),
    max_tokens=0,
    temperature=0,
    top_p=0.95,
    top_k=5,
    repeat_penalty=1.0
  )
print(ans)
print("completion_tokens = ", moutput['usage']['completion_tokens'])

Llama.generate: prefix-match hit


 There are several possible causes for sudden, patchy hair loss, also known as alopecia areata. Here are some effective treatments and possible causes:

Causes:
1. Alopecia Areata: An autoimmune disorder that causes the body's immune system to attack hair follicles, leading to hair loss.
2. Stress: Physical or emotional stress can cause hair loss.
3. Nutritional Deficiencies: Lack of certain nutrients, such as iron, zinc, or biotin, can lead to hair loss.
4. Hormonal Imbalance: Hormonal changes, such as those caused by pregnancy, menopause, or thyroid problems, can cause hair loss.
5. Medications: Certain medications, such as chemotherapy drugs, can cause hair loss.

Treatments:
1. Minoxidil: A topical medication that can help stimulate hair growth and slow down hair loss.
2. Corticosteroids: Prescription medications that can help reduce inflammation and suppress the immune system to promote hair growth.
3. Immunotherapy: Injections of certain proteins that can help stimulate hair grow

In [ ]:
# top_k set to 70
system_prompt = ""

ans, moutput = generate_response(
    prepare_model_prompt(system_prompt, query_3),
    max_tokens=0,
    temperature=0,
    top_p=0.95,
    top_k=70,
    repeat_penalty=1.0
  )
print(ans)
print("completion_tokens = ", moutput['usage']['completion_tokens'])

Llama.generate: prefix-match hit


 There are several possible causes for sudden, patchy hair loss, also known as alopecia areata. Here are some effective treatments and possible causes:

Causes:
1. Alopecia Areata: An autoimmune disorder that causes the body's immune system to attack hair follicles, leading to hair loss.
2. Stress: Physical or emotional stress can cause hair loss.
3. Nutritional Deficiencies: Lack of certain nutrients, such as iron, zinc, or biotin, can lead to hair loss.
4. Hormonal Imbalance: Hormonal changes, such as those caused by pregnancy, menopause, or thyroid problems, can cause hair loss.
5. Medications: Certain medications, such as chemotherapy drugs, can cause hair loss.

Treatments:
1. Minoxidil: A topical medication that can help stimulate hair growth and slow down hair loss.
2. Corticosteroids: Prescription medications that can help reduce inflammation and suppress the immune system to promote hair growth.
3. Immunotherapy: Injections of certain proteins that can help stimulate hair grow

Observation

While the “Causes” sections remain identical, the “Treatments” sections differ significantly based on the top_k setting.

With top_k = 70, the response includes a more comprehensive list of treatments, improved specificity in wording, and a higher overall token count.

This variation occurs because top_k = 5 restricts the model to selecting the next word from only the top five most probable candidates. As a result, the output tends to be more predictable, concise, and generic.

In contrast, top_k = 70 expands the candidate pool to seventy possible words at each step. This broader selection allows the model to incorporate more precise terminology, generate richer detail, and produce a more extensive and nuanced response.

# **Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?**
Combination 4 - Few-shot prompting

In [ ]:
system_prompt = """
You are a medical assistant providing information on treatments for brain injuries.

User:
Question: What are the common symptoms and treatments for pulmonary embolism?
Answer: Common symptoms of pulmonary embolism include sudden shortness of breath, chest pain that worsens with breathing or coughing, rapid heart rate, rapid breathing, anxiety, coughing (sometimes with blood), sweating, and fainting. Treatment typically involves anticoagulant medications to prevent further clots, and sometimes thrombolytics to dissolve existing clots. In severe cases, surgical embolectomy or catheter-directed treatments may be necessary.

User:
Question: Can you provide the trade names of medications used for treating hypertension?
Answer: Some common trade names for medications used to treat hypertension include Prinivil, Zestril (Lisinopril), Norvasc (Amlodipine), Cozaar (Losartan), Diovan (Valsartan), Toprol XL, Lopressor (Metoprolol), and Tenormin (Atenolol).

User:
Question: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
Answer:
"""

user_input = ""

ans, moutput = generate_response(
    prepare_model_prompt(system_prompt, user_input),
    max_tokens=0
  )
print(ans)
print("completion_tokens = ", moutput['usage']['completion_tokens'])

Llama.generate: prefix-match hit


 Treatment for a brain injury can depend on the severity and location of the injury. For mild to moderate brain injuries, rest, medication for pain and swelling, and rehabilitation therapies such as physical, occupational, and speech therapy may be recommended. For more severe injuries, treatments may include surgery to remove hematomas or repair skull fractures, and intensive care to manage symptoms such as seizures, infections, or breathing problems. Rehabilitation is also an important part of treatment for brain injuries, regardless of severity. It can help individuals regain skills and improve function. Additionally, medications may be prescribed to manage symptoms such as seizures, depression, or difficulty with attention or memory. It's important to note that every brain injury is unique, and treatment plans will vary depending on the individual's specific needs.
completion_tokens =  174


# **Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?**

Combination 5 - Chain-of-Thought prompting

In [ ]:
system_prompt = """Think step-by-step to determine the necessary precautions, treatment steps, and considerations for care and recovery for a person who has fractured their leg during a hiking trip. Consider the immediate actions to take at the injury site, the subsequent medical treatment, and the long-term recovery process.
"""

ans, moutput = generate_response(
    prepare_model_prompt(system_prompt, query_5),
    max_tokens=0
  )
print(ans)
print("completion_tokens = ", moutput['usage']['completion_tokens'])

Llama.generate: prefix-match hit


 I. Immediate Actions at the Injury Site:
1. Assess the situation: Check if the person is in a safe location and if there are any other injuries.
2. Provide first aid: Apply a sterile dressing to the wound, if present, to prevent infection. Do not attempt to realign the bone or apply excessive pressure to the area.
3. Immobilize the leg: Use a splint, a makeshift sling, or a hiking pole to immobilize the leg to prevent further damage and provide comfort.
4. Monitor vital signs: Check for signs of shock, such as rapid heartbeat, shallow breathing, or pale skin.
5. Provide hydration and nutrition: Offer water or other fluids to help maintain hydration and provide energy-rich snacks.

II. Subsequent Medical Treatment:
1. Seek professional help: Arrange for transportation to the nearest medical facility as soon as possible.
2. Diagnostic tests: X-rays will be used to confirm the fracture and determine the extent of the injury.
3. Pain management: The healthcare provider may prescribe pain 

Observation

The response is detailed and demonstrates structured, step-by-step reasoning in its explanation.

# **4 - Download Embedding model**


Download the General Text Embeddings (GTE) model to generate embeddings for the PDF data from the Merck Manual.

These models are ranked well on the MTEB Leaderboard for 'Retrieval' tasks, indicating their effectiveness in creating meaningful representations of text for search and retrieval purposes.
This model exclusively caters to English texts, and any lengthy texts will be truncated to a maximum of 512 tokens.

In [ ]:
# Import the SentenceTransformerEmbeddings class for creating sentence embeddings.
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

In [ ]:
from langchain_community.embeddings import SentenceTransformerEmbeddings


In [ ]:
# Uninstall potentially conflicting numpy and sentence-transformers  #Please don't run this query  #Please don't run this query
!pip uninstall -y numpy sentence-transformers

# Install a compatible version of numpy
!pip install numpy==1.26.4

# Reinstall sentence-transformers to ensure compatibility with the new numpy
!pip install sentence-transformers

#Load the GTE-Large embedding model
embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")

In [ ]:
embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")

In [ ]:
print("Model information:")
print(embedding_model.client)

print("\nTokenizer:")
print(embedding_model.client.tokenizer)

Model information:
SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

Tokenizer:
BertTokenizer(name_or_path='thenlper/gte-large', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=Fal

# **5 - Data Preparation and Vector Database Setup for RAG**

To prepare the medical manual data for Retrieval Augmented Generation (RAG), we will perform the following steps:


Chunking: Divide the PDF document into smaller, manageable text segments (chunks).
We will create two sets of chunks with different sizes (490 and 245 tokens) to explore the impact of chunk size on retrieval.

Vectorization: Convert these text chunks into numerical representations called embeddings using the pre-trained GTE-Large embedding model.

Vector Database Setup: Store the vectorized chunks in two separate Chroma vector databases, one for each chunk size. This allows for efficient similarity search during the retrieval phase of RAG.

By creating two databases with different chunk sizes, we can compare their effectiveness in retrieving relevant information for answering medical queries.

# **5.2 - Import libraries required for chunking**

In [ ]:
# Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

# Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import Chroma
from uuid import uuid4
from time import sleep

# **5.2 - Loading and Previewing the Medical Manual**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#manual_pdf_path = "/content/drive/MyDrive/Colab Notebooks/Project-5/medical_diagnosis_manual.pdf"

import pandas as pd
manual_pdf_path  = '/content/drive/MyDrive/medical_diagnosis_manual.pdf'

In [ ]:
pdf_loader = PyMuPDFLoader(manual_pdf_path)

In [ ]:
manual = pdf_loader.load()

In [ ]:
print("total documents loaded from the PDF = ", len(manual))

total documents loaded from the PDF =  4114


In [ ]:
# Inspect page_content and length of few randomly selected documents to understand the data structure
for i in range(20,24):
  print("page = ",manual[i].metadata['page'],end="\n")
  print("page_content length = ", len(manual[i].page_content),end="\n")
  print("---"*10)

page =  20
page_content length =  1489
------------------------------
page =  21
page_content length =  1845
------------------------------
page =  22
page_content length =  1860
------------------------------
page =  23
page_content length =  1799
------------------------------


# **5.3 - Define utility function get_vectordb_handler to populate vector DB**

In [ ]:
# Define a utility function to create and populate database
def get_vectordb_handler(persist_dir, collection_name, document_chunks):
    """
    Handles the creation or loading of the Chroma vector database.

    Args:
        persist_dir (str): The directory path to persist the database.
        collection_name (str): The name of the collection within the database.
        document_chunks (list): A list of document chunks to add to the database.

    Returns:
        Chroma: An instance of the Chroma vector database.
    """
    if os.path.exists(persist_dir):
      print(f'"{persist_dir}" already exists!')
    else:
      print(f'Creating vector database directory in "{persist_dir}"')
      os.makedirs(persist_dir)

    # Instantiate Chroma with persitence
    vectorstore = Chroma(
        persist_directory=persist_dir,
        embedding_function=embedding_model,
        collection_name=collection_name
      )

    # Get the collection
    content = vectorstore.get()

    if not len(content['ids']):
      print(f'Populating vector database...')

      uuids = [str(uuid4()) for _ in range(len(document_chunks))]
      i = 0
      while i < len(document_chunks) - 1000:
        added_list = vectorstore.add_documents(document_chunks[i : i + 1000], ids=uuids[i : i + 1000])
        print(f'Vector database populated with {len(added_list)} entries')
        i += 1000
        sleep(10)

      if i < len(document_chunks):
          added_list = vectorstore.add_documents(document_chunks[i :], ids=uuids[i :])
          print(f'Vector database populated with {len(added_list)} entries')

    else:
      print(f'Vector database already populated.')

    return vectorstore

# **5.3 - Data Chunking (chunk_size=490)**

In [ ]:
# Import the BertTokenizerFast from the transformers library
from transformers import BertTokenizerFast
# Load the tokenizer for the 'thenlper/gte-large' model
tokenizer = BertTokenizerFast.from_pretrained("thenlper/gte-large")

In [ ]:
# Initialize the RecursiveCharacterTextSplitter using the loaded tokenizer.
# from_huggingface_tokenizer is used to ensure compatibility with the model's tokenizer.
# chunk_size is set to 490 : The maximum number of tokens in each chunk
# chunk_overlap is set to 20 : The number of tokens to overlap between consecutive chunks
text_splitter_490 = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=490,
    chunk_overlap=20
)

In [ ]:
# Load the PDF document and split it into chunks using the configured text_splitter
document_chunks = pdf_loader.load_and_split(text_splitter_490)

In [ ]:
# Verify token counts for each chunk
max_tokens_allowed = 512
all_chunks_within_limit = True

for i, chunk in enumerate(document_chunks):
  token_count = len(tokenizer.encode(chunk.page_content))
  if token_count > max_tokens_allowed:
    print(f"Chunk {i} exceeds the token limit with {token_count} tokens.")
    all_chunks_within_limit = False

if all_chunks_within_limit:
  print(f"All document chunks are within the {max_tokens_allowed}-token limit.")

All document chunks are within the 512-token limit.


In [ ]:
# Print the total number of document chunks created
print(f"""
type(document_chunks) = {type(document_chunks)}
type(document_chunks[0]) = {type(document_chunks[0])}
len(document_chunks) = {len(document_chunks)}""")


type(document_chunks) = <class 'list'>
type(document_chunks[0]) = <class 'langchain_core.documents.base.Document'>
len(document_chunks) = 8628


In [ ]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [ ]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

# **5.4 - Populate Vector Database (medical_db_490)**

In [ ]:
# Define the directory where the vector database will be stored
persist_dir  = '/content/drive/MyDrive/medical_db_490'



In [ ]:
vectorstore = get_vectordb_handler(persist_dir, "MerckManual", document_chunks)

"/content/drive/MyDrive/medical_db_490" already exists!


/tmp/ipython-input-3842410633.py:21: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


Vector database already populated.


In [ ]:
# Total entries in the vector db
len(vectorstore.get()['ids'])

8628

In [ ]:
# Test similarity search for vitamin A toxicity
vectorstore.similarity_search("What are the side effects if vitamin A overdose?",k=3)

[Document(metadata={'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'creationdate': '2012-06-15T05:44:40+00:00', 'modDate': 'D:20260205230753Z', 'file_path': '/content/drive/MyDrive/medical_diagnosis_manual.pdf', 'page': 93, 'source': '/content/drive/MyDrive/medical_diagnosis_manual.pdf', 'author': '', 'subject': '', 'creator': 'Atop CHM to PDF Converter', 'total_pages': 4114, 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'keywords': '', 'creationDate': 'D:20120615054440Z', 'moddate': '2026-02-05T23:07:53+00:00', 'format': 'PDF 1.7', 'trapped': ''}, page_content='defects occur in children of women receiving isotretinoin (which is related to vitamin A) for acne treatment\nduring pregnancy.\nAlthough carotene is converted to vitamin A in the body, excessive ingestion of carotene causes\ncarotenemia, not vitamin A toxicity. Carotenemia is usually asymptomatic but may lead to carotenodermia,\nin which the skin becomes yellow. When taken as a supplement, β-ca

Observation

The Merck Manuals content has been successfully vectorized and stored in the ChromaDB vector database.

A total of 8,628 entries are present in the database, corresponding directly to the number of document chunks generated during preprocessing.

A similarity search performed using the query “Vitamin A toxicity” successfully retrieved relevant document chunks, confirming that the embedding and retrieval pipeline is functioning as expected.

# **5.5 - Data Chunking (chunk_size=245)**



To tune chunking, we'll create a new database to store smaller size chunks.

In [ ]:
# Initialize the RecursiveCharacterTextSplitter for chunk_size 245 (smaller than the previous one)
text_splitter_245 = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=245,
    chunk_overlap=20
)

In [ ]:
# Load the PDF document and split it into chunks using the configured text_splitter_245
document_chunks_245 = pdf_loader.load_and_split(text_splitter_245)

In [ ]:
# Print the total number of document chunks created using text_splitter_245 and their types
print(f"""
type(document_chunks_245) = {type(document_chunks_245)}
type(document_chunks_245[0]) = {type(document_chunks_245[0])}
len(document_chunks_245) = {len(document_chunks_245)}""")


type(document_chunks_245) = <class 'list'>
type(document_chunks_245[0]) = <class 'langchain_core.documents.base.Document'>
len(document_chunks_245) = 16086


In [ ]:
# Print the dimension of the embedding vector generated by the model.
print("Dimension of the embedding vector ",len(embedding_model.embed_query(document_chunks_245[0].page_content)))

Dimension of the embedding vector  1024


Observation

The embedded vector dimension remains at 1024, which aligns with the embedding model’s fixed output size.

This indicates that even when using smaller chunks (chunk_size = 245), the embedding model successfully captures the contextual meaning of each chunk and encodes it into a consistent 1024-dimensional vector representation.

# **5.6 - Populate Vector Database (medical_db_245)**

In [ ]:
# Define the directory where the vector database will be stored
persist_dir_245 = '/content/drive/MyDrive/medical_db_245'

In [ ]:
vectorstore_245 = get_vectordb_handler(persist_dir_245, "MerckManual245", document_chunks_245)

"/content/drive/MyDrive/medical_db_245" already exists!
Vector database already populated.


In [ ]:
# Total entries in the vector db
len(vectorstore_245.get()['ids'])

16086

In [ ]:
# Test similarity search for vitamin A toxicity
vectorstore_245.similarity_search("What are the side effects of vitamin A overdose?",k=3)

[Document(metadata={'page': 93, 'keywords': '', 'subject': '', 'total_pages': 4114, 'author': '', 'file_path': '/content/drive/MyDrive/medical_diagnosis_manual.pdf', 'creationDate': 'D:20120615054440Z', 'trapped': '', 'modDate': 'D:20260205230753Z', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'creationdate': '2012-06-15T05:44:40+00:00', 'format': 'PDF 1.7', 'creator': 'Atop CHM to PDF Converter', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'moddate': '2026-02-05T23:07:53+00:00', 'source': '/content/drive/MyDrive/medical_diagnosis_manual.pdf'}, page_content='are present, adjusting the dose almost always leads to complete recovery.\nAcute vitamin A toxicity in children may result from taking large doses (> 100,000 RAE [> 300,000 IU]),\nusually accidentally. In adults, acute toxicity has occurred when arctic explorers ingested polar bear or\nseal livers, which contain several million units of vitamin A.\nChronic toxicity in older children and adults u

# **5.7 - Tranform vectorstore into retriever**
For easier usage with LangChain chains, we can tranform the vector store into retriever

In [ ]:

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [ ]:
rel_docs = retriever.get_relevant_documents("What are the side effects if vitamin A overdose?")
rel_docs

[Document(metadata={'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'source': '/content/drive/MyDrive/medical_diagnosis_manual.pdf', 'format': 'PDF 1.7', 'moddate': '2026-02-05T23:07:53+00:00', 'creationdate': '2012-06-15T05:44:40+00:00', 'creator': 'Atop CHM to PDF Converter', 'creationDate': 'D:20120615054440Z', 'keywords': '', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'page': 93, 'total_pages': 4114, 'trapped': '', 'modDate': 'D:20260205230753Z', 'author': '', 'file_path': '/content/drive/MyDrive/medical_diagnosis_manual.pdf', 'subject': ''}, page_content='defects occur in children of women receiving isotretinoin (which is related to vitamin A) for acne treatment\nduring pregnancy.\nAlthough carotene is converted to vitamin A in the body, excessive ingestion of carotene causes\ncarotenemia, not vitamin A toxicity. Carotenemia is usually asymptomatic but may lead to carotenodermia,\nin which the skin becomes yellow. When taken as a supplement, β-ca

Observation

You can observe that the same documents were retrieved for the identical query using the retriever, indicating consistent retrieval behavior for repeated inputs.

In [ ]:
def generate_response(prompt, max_tokens=200):
    response = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=0.7
    )
    return response["choices"][0]["text"], response


In [ ]:
# Query Mistral-7B-Instruct without retrieved context
rsp, mo = generate_response("What are the side effects of vitamin A overdose?", max_tokens=0)

In [ ]:
print(rsp)



Vitamin A is an essential nutrient that plays a crucial role in maintaining good vision, skin health, immune function, and cell growth. However, consuming too much vitamin A can lead to toxicity, also known as hypervitaminosis A. Symptoms of vitamin A overdose may include:

1. Nausea and vomiting
2. Dizziness and headaches
3. Fatigue and weakness
4. Dry skin and hair loss
5. Joint pain and muscle weakness
6. Liver damage (in severe cases)
7. Birth defects in developing fetuses (if taken during pregnancy)
8. Skin irritation, itching, and peeling
9. Bone pain and fractures
10. Confusion and memory loss

It is essential to note that vitamin A toxicity is rare, as our bodies can store excess amounts of this nutrient in the liver. However, taking high doses of vitamin A supplements or consuming large amounts of foods rich in vitamin A, such as liver, fortified dairy products, and sweet potatoes, can increase the risk of overdose.

The recommended daily intake of vitamin A for adults is 70

Observation

The above response is generic in nature and appears to be generated based primarily on the model’s pre-trained knowledge, rather than being grounded in the retrieved content from the medical manual.

# **5.8 - Define utility function prepare_rag_model_prompt**

Prompts guide the model to generate accurate responses.

1. The system prompt describing the assistant's role.
2. A user message includes context and the question.

In [ ]:
# Define a simple utility function to prepare model prompt for RAG
def prepare_rag_model_prompt(
    system_prompt,
    query,
    retriever,
    k=3
):
    # Retrieve relevant document chunks from retriever
    relevant_docs = retriever.get_relevant_documents(query=query, k=k)
    context = [d.page_content for d in relevant_docs]

    # Combine the retrieved documents into one long string
    context_string = ". ".join(context)

    user_prompt = """
    ###Context
    Here are the retrieved documents that are releavnt to the question mentioned below.
    {context_string}

    ###Question
    {query}
    """.format(context_string=context_string, query=query)

    # Return the prepared prompt and the context string
    return (
        f"""<s>[INST]{'system'}: {system_prompt}
                {'user'}: {user_prompt}
                [/INST]""" ,
        context_string
    )

# **5.9 - RAG query**

In [ ]:
# Query Mistral-7B-Instruct with retrieved context
system_message = """
You are an assistant whose work is to review the report and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.

If the answer is not found in the context, strictly respond "I don't know".
"""

In [ ]:
user_question = "What are the side effects if vitamin A overdose?"

In [ ]:
prompt, _ = prepare_rag_model_prompt(system_message, user_question, retriever)

In [ ]:
rsp, mo = generate_response(prompt,max_tokens=0)
print(rsp)

Llama.generate: prefix-match hit


 The context does not mention specific side effects for vitamin A overdose beyond those related to toxicity, which include headache, increased intracranial pressure, changes in skin, hair, and nails, abnormal liver test results, and, in a fetus, birth defects.


Observation

The response correctly retrieves and utilizes information grounded specifically in the provided context (Merck Manuals), demonstrating effective context-aware generation.

# 6 - Question Answering using RAG
Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
# context window is set to 5000
# Each chunk is 512; set k = 3 to retrieve 3 top matching chunks
print("query1 : ", query_1)
rag_prompt, _ = prepare_rag_model_prompt(system_message, query_1, retriever, k=3)
mrq1, moq1 = generate_response(rag_prompt, max_tokens=0)
print(mrq1)

query1 :  What is the protocol for managing sepsis in a critical care unit?


Llama.generate: prefix-match hit


 The protocol for managing sepsis in a critical care unit includes giving parenteral antibiotics after specimens have been taken for Gram stain and culture, starting very prompt empiric therapy immediately after suspecting sepsis, and changing the antibiotic regimen based on culture and sensitivity results. One suggested regimen is gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin, or ceftazidime plus a fluoroquinolone if Pseudomonas is suspected. Vancomycin should be added if resistant staphylococci or enterococci are suspected, and a drug effective against anaerobes should be included if there is an abdominal source. Antibiotics are continued for at least 5 days after shock resolves and evidence of infection subsides. Abscesses must be drained, and necrotic tissues must be surgically excised to eliminate septic foci. Normalization of blood glucose improves outcome in critically ill patients, and a continuous IV insulin infusion is titrated to maintain

# **Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?**

In [ ]:
print("query2 : ", query_2)
rag_prompt, _ = prepare_rag_model_prompt(system_message, query_2, retriever, k=3)
mrq2, moq2 = generate_response(rag_prompt, max_tokens=0)
print(mrq2)

query2 :  What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?


Llama.generate: prefix-match hit


 The common symptoms of appendicitis include abdominal pain, anorexia, and abdominal tenderness. Appendicitis cannot be cured via medicine alone as it requires surgical removal. Therefore, the appropriate surgical procedure for treating appendicitis is an appendectomy.


# **Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?**

In [ ]:
print("query3 : ", query_3)
rag_prompt, _ = prepare_rag_model_prompt(system_message, query_3, retriever, k=3)
mrq3, moq3 = generate_response(rag_prompt, max_tokens=0)
print(mrq3)

query3 :  What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?


Llama.generate: prefix-match hit


 Sudden patchy hair loss, also known as alopecia areata, can be treated with various methods. Topical corticosteroids, intralesional or systemic, minoxidil, anthralin, immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA) can be effective treatments. The cause of this condition is believed to be an autoimmune disorder, affecting genetically susceptible individuals exposed to unclear environmental triggers. In severe cases, oral antimalarials, corticosteroids, retinoids, or immunosuppressants may also be considered. It is important to note that treatment outcomes and response times vary from person to person. If you have concerns about your hair loss or suspect alopecia areata, it is recommended to consult with a healthcare professional for an accurate diagnosis and personalized treatment plan.


**Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?**

In [ ]:
print("query4 : ", query_4)
rag_prompt, _ = prepare_rag_model_prompt(system_message, query_4, retriever, k=3)
mrq4, moq4 = generate_response(rag_prompt, max_tokens=0)
print(mrq4)

query4 :  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?


Llama.generate: prefix-match hit


 The recommended treatments for a person with a physical injury to brain tissue that temporarily or permanently impairs brain function include ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation. Supportive care should include preventing systemic complications due to immobilization, providing good nutrition, and preventing pressure ulcers. Rehabilitation through a team approach combining physical, occupational, and speech therapy, skill-building activities, and counseling may be necessary for patients whose coma exceeds 24 ho

# **Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?**

In [ ]:
print("query5 : ", query_5)
rag_prompt, _ = prepare_rag_model_prompt(system_message, query_5, retriever, k=3)
mrq5, moq5 = generate_response(rag_prompt, max_tokens=0)
print(mrq5)

query5 :  What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?


Llama.generate: prefix-match hit


 The context mentions that a person with a fractured leg should keep the cast dry, never put an object inside the cast, inspect the cast's edges and skin around it every day, apply lotion to any red or sore areas, pad any rough edges, seek medical care if there is an odor from within the cast or a fever, maintain good hygiene, and follow instructions for using a splint instead of a cast if possible. The person should also be immobilized initially, but prolonged immobilization can lead to complications such as stiffness, contractures, and muscle atrophy. In case of a fracture, treatment involves analgesics, immobilization, and sometimes surgery. Prehospital care for frostbitten extremities includes rapid rewarming in warm water and keeping the body warm. Acute care involves stabilizing core temperature and rapidly rewarming the affected area in a hospital setting.


Observation

The responses to Queries 1 through 5 are now generated using information retrieved from the Merck Manuals, demonstrating the effectiveness of the RAG approach in grounding answers in relevant and authoritative source material.

# **6.1 - Fine-tuning the RAG System (chunking, retriever, LLM parameters)**

In [ ]:
# Define retriever for the vector database storing chunks of 245 tokens
retriever_245 = vectorstore_245.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

# Query 1: What is the protocol for managing sepsis in a critical care unit?
Combination 1 - System prompt to refer context and use retriever_245

In [ ]:
# Each chunk is 245; set k = 2 to only retrieve 2 similar chunks
print("query1 : ", query_1)
rag_prompt, _ = prepare_rag_model_prompt(system_message, query_1, retriever_245, k=2)
mrq1, moq1 = generate_response(rag_prompt, max_tokens=0)
print(mrq1)

query1 :  What is the protocol for managing sepsis in a critical care unit?


Llama.generate: prefix-match hit


 Patients with sepsis in a critical care unit should be monitored frequently for systemic pressure, CVP or PAOP, pulse oximetry, ABGs, blood glucose, lactate, electrolyte levels, renal function, sublingual PCO2, urine output, and possibly echocardiography to identify limitations in left ventricular function and incipient pulmonary edema due to fluid overload. Fluid resuscitation with 0.9% saline should be given until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg. Oliguria with hypotension is not a contraindication to vigorous fluid resuscitation, and the quantity of fluid required often exceeds the normal blood volume and may reach 10 L over 4 to 12 hours. Supportive care includes adequate nutrition, prevention of infection, stress ulcers and gastritis, and pulmonary embolism.


Observation

Setting the number of retrieved documents (k) to 2 when using retriever_245 provides less contextual information to the LLM compared to using a retriever configured with k = 3.

As a result, the model has access to a narrower context window, which may impact the completeness and depth of the generated response.

# **Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?**
Combination 2 - System prompt to refer context and use retriever_245 with k=7

In [ ]:
print("query2 : ", query_2)
rag_prompt, _ = prepare_rag_model_prompt(system_message, query_2, retriever_245, k=7)
mrq2, moq2 = generate_response(rag_prompt,max_tokens=0)
print(mrq2)

query2 :  What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?


Llama.generate: prefix-match hit


 The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which shifts to the right lower quadrant after a few hours. Pain increases with cough and motion, and there may be direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Other signs include pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), and an increase in pain from passive extension of the right hip joint. Appendicitis cannot be cured via medicine alone, and the standard treatment is surgical removal of the appendix through open or laparoscopic appendectomy.


Observation
Setting the number of retrieved documents (k) to 7 when using retriever_245 provides a broader context to the LLM compared to using a retriever configured with k = 3.

As a result, the model generates a more detailed and comprehensive response due to the increased contextual information available during generation.

# **Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?**
Combination 3 - System prompt to refer context and use retriever_245 with temperature = 1

In [ ]:
print("query3 : ", query_3)
rag_prompt, _ = prepare_rag_model_prompt(system_message, query_3, retriever_245, k=3)
mrq3, moq3 = generate_response(rag_prompt, max_tokens=0)
print(mrq3)

query3 :  What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?


Llama.generate: prefix-match hit


 Sudden patchy hair loss, also known as alopecia areata, can be treated with various options including topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). The cause behind this condition is not clear but can be linked to genetics and stress. Other possible causes for sudden hair loss include nutritional deficiencies, thyroid issues, or autoimmune diseases. It's important to consult a healthcare professional for an accurate diagnosis and treatment plan.


Observation
Setting the temperature to 1 introduces greater randomness into the response generation process, resulting in more varied and less deterministic outputs.

**Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?**
Combination 4 - System prompt to refer context and use retriever_245 with max_tokens = 100

In [ ]:
print("query4 : ", query_4)
rag_prompt, _ = prepare_rag_model_prompt(system_message, query_4, retriever_245, k=3)
mrq4, moq4 = generate_response(rag_prompt,max_tokens=0)
print(mrq4)
print("completion_tokens = ", moq4['usage']['completion_tokens'])

query4 :  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?


Llama.generate: prefix-match hit


 The text suggests that supportive care is recommended for a person with a physical injury to brain tissue. This care includes preventing systemic complications due to immobilization, providing good nutrition, and preventing pressure ulcers. There is no mention of specific treatments for the brain injury itself in the context provided.
completion_tokens =  60


Observation

The number of completion tokens is fewer than 100, which is consistent with the max_tokens = 100 parameter defined in the generate_response function.

# **Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?**
Combination 5 - System prompt to respond in bullet list and use retriever_245 with k=3

In [ ]:
sys = system_message + """
Respond in exactly 4 bullet points.
"""

In [ ]:
print("query5 : ", query_5)
rag_prompt, _ = prepare_rag_model_prompt(sys, query_5, retriever_245, k=3)
mrq5, moq5 = generate_response(rag_prompt, max_tokens=0)
print(mrq5)

query5 :  What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?


Llama.generate: prefix-match hit


 * The person with a fractured leg should seek medical care at once.
* In the meantime, they should apply ice to minimize swelling and pain, and rest the affected leg to prevent further injury.
* The leg should be immobilized using a splint or cast to allow for proper healing and to maintain elevation above heart level when feasible.
* Patients may need analgesia or sedation for definitive treatment which usually involves reduction, either closed or open, depending on the severity of the fracture. After treatment, early mobilization is recommended to minimize contractures and muscle atrophy.


# **7 - Output Evaluation**
Evaluation of the RAG system will be performed using the LLM-as-a-judge method. This is an effective method when human annotated/referece text is not available as gold standard reference.


This Notebook uses the same Mistral model to evaluate the quality of the RAG system's responses based on two key aspects:


Faithfulness (also called Hallucination rate - inversely related): Measures how well the generated response aligns with retrieved documents, avoiding hallucinations.

Assessing Relevance: How well the system uses the retrieved information to generate accurate and helpful answers.

The Llama 2 13B (trained on 13 billion parameters) model will be downloaded and loaded for this evaluation. Note that this model is approximately 11GB in size.

# **7.2 - Define utility function generate_faithfulness_and_relevance_score**

In [ ]:
def generate_faithfulness_and_relevance_score(
    user_input, # user query
    system_prompt, # prompt to generate RAG response
    retriever, # DB retriever
    faithfulness_rater_system_message,
    relevance_rater_system_message,
    max_tokens=0,
    temperature=0,
    top_p=0.95,
    top_k=50,
    repeat_penalty=1.0,
    k=3,
):
    rag_prompt, context_retrieved = prepare_rag_model_prompt(
        system_prompt,
        user_input,
        retriever,
        k=k
    )

    rag_response, model_output = generate_response(
        rag_prompt,
        max_tokens=max_tokens
    )

    message_template = """
      ###Question
      {question}

      ###Context
      {context}

      ###Answer
      {answer}
      """

    # Generate faithfulness_prompt
    faithfulness_prompt = f"""<s>[INST]{faithfulness_rater_system_message}\n
                {'user'}: {message_template.format(question=user_input,context=context_retrieved, answer=rag_response)}
                [/INST]"""

    #print("faithfulness_prompt = ", faithfulness_prompt)

    # Generate relevance_prompt
    relevance_prompt = f"""<s>[INST]{relevance_rater_system_message}\n
                {'user'}: {message_template.format(question=user_input,context=context_retrieved, answer=rag_response)}
                [/INST]"""

    #print("relevance_prompt = ", relevance_prompt)

    faith_response = llm(
            prompt=faithfulness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    relevance_response = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    return faith_response['choices'][0]['text'],relevance_response['choices'][0]['text']

In [ ]:
faithfulness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
Rate it 1 - if The ###Answer is not derived from the ###Context at all.
Rate it 2 - if The ###Answer is derived from the ###Context only to a limited extent.
Rate it 3 - if The ###Answer is derived from ###Context to a good extent.
Rate it 4 - if The ###Answer is derived from ###Context mostly.
Rate it 5 - if The ###Answer is derived from ###Context completely.

Metric:
The answer should be derived only from the information presented in the context.

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.

Please note: Make sure you give a single overall rating in the range of 1 to 5 along with an overall explanation.
"""

In [ ]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
- Rate 1 – The ###Answer is not relevant to the ###Question at all.
- Rate 2 – The ###Answer is only slightly relevant to the **###Question**, missing key aspects.
- Rate 3 – The ###Answer is moderately relevant, addressing some parts of the **###Question** but leaving out important details.
- Rate 4 – The ###Answer is mostly relevant, covering key aspects but with minor gaps.
- Rate 5 – The ###Answer is fully relevant, directly answering all important aspects of the **###Question** with appropriate details from the **###Context**.

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.

Note: Provide a single overall rating in the range of 1 to 5, along with a brief explanation of why you assigned that score.
"""


# **Query 1: What is the protocol for managing sepsis in a critical care unit?**

In [ ]:
ground,rel = generate_faithfulness_and_relevance_score(
    user_input=query_1,
    system_prompt=system_message,
    retriever=retriever,
    faithfulness_rater_system_message=faithfulness_rater_system_message,
    relevance_rater_system_message=relevance_rater_system_message
)
print("Faithfulness metric : \n", ground,end="\n\n\n\n")
print("Relevance metric : \n",rel)


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Faithfulness metric : 
  To evaluate the answer, we need to follow these steps:

1. Identify the key information related to managing sepsis in a critical care unit from the context.
2. Compare the AI generated answer with the identified key information.
3. Determine the extent to which the AI generated answer is derived from the context.

The context provides detailed information about the approach to managing critically ill patients, including sections on patient monitoring and testing, blood tests, and treatment of sepsis. The key information related to managing sepsis in a critical care unit includes:

1. Administration of antibiotics such as gentamicin or tobramycin, along with a third-generation cephalosporin, ceftazidime, or a fluoroquinolone.
2. Addition of vancomycin if resistant staphylococci or enterococci are suspected.
3. Inclusion of a drug effective against anaerobes if there is an abdominal source.
4. Continuation of antibiotics for at least 5 days after shock resolves a

# **Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?**

In [ ]:
ground,rel = generate_faithfulness_and_relevance_score(
    user_input=query_2,
    system_prompt=system_message,
    retriever=retriever,
    faithfulness_rater_system_message=faithfulness_rater_system_message,
    relevance_rater_system_message=relevance_rater_system_message
)

print("Faithfulness metric : \n", ground,end="\n\n\n\n")
print("Relevance metric : \n",rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Faithfulness metric : 
  Steps to evaluate the answer:
1. Identify the key information in the context related to appendicitis and its treatment.
2. Determine if the symptoms mentioned in the question are present in the context.
3. Check if the treatment options given in the answer are consistent with the context.
4. Verify if the answer mentions only the information derived from the context.

Explanation:
The answer correctly identifies the common symptoms of appendicitis as abdominal pain, anorexia, and abdominal tenderness. It also states that appendicitis cannot be cured via medicine alone and requires surgical removal for treatment. The surgical procedure mentioned for treating appendicitis is appendectomy, which is consistent with the context. Therefore, the answer appears to be derived solely from the information presented in the context.

Rating: 5 (The AI generated answer is derived completely from the context)



Relevance metric : 
  Steps to evaluate the context:
1. Identify

# **Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?**

In [ ]:
ground,rel = generate_faithfulness_and_relevance_score(
    user_input=query_3,
    system_prompt=system_message,
    retriever=retriever,
    faithfulness_rater_system_message=faithfulness_rater_system_message,
    relevance_rater_system_message=relevance_rater_system_message
)

print("Faithfulness metric : \n", ground,end="\n\n\n\n")
print("Relevance metric : \n",rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Faithfulness metric : 
  Steps to evaluate the answer:
1. Identify the main topic of the question and context. In this case, it is about sudden patchy hair loss and its causes and treatments.
2. Determine if the AI generated answer directly addresses the question and is derived only from the information presented in the context.
3. Check if the AI generated answer mentions any specific treatments or solutions for sudden patchy hair loss that are mentioned in the context.
4. Verify if the possible causes of sudden patchy hair loss mentioned in the AI generated answer are consistent with those in the context.

Explanation:
The AI generated answer directly addresses the question by mentioning several treatment options for sudden patchy hair loss, which is the main topic of the question. The treatments mentioned in the answer, such as topical and systemic corticosteroids, minoxidil, anthralin, immunotherapy, and PUVA, are all mentioned in the context as possible treatments for alopecia are

# **Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?**

In [ ]:
ground,rel = generate_faithfulness_and_relevance_score(
    user_input=query_4,
    system_prompt=system_message,
    retriever=retriever,
    faithfulness_rater_system_message=faithfulness_rater_system_message,
    relevance_rater_system_message=relevance_rater_system_message
)

print("Faithfulness metric : \n", ground,end="\n\n\n\n")
print("Relevance metric : \n",rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Faithfulness metric : 
  To evaluate the answer given to the question as per the metric, the following steps need to be taken:

1. Identify the key information in the context that relates to the recommended treatments for a person with a brain injury.
2. Determine if the AI generated answer includes all the necessary information identified in step 1.
3. Check if the information in the answer is derived only from the context and not from any external sources.

Based on the given context, the key information related to recommended treatments for a person with a brain injury includes:
- Supportive care such as preventing systemic complications, providing good nutrition, and preventing pressure ulcers.
- Rehabilitation including physical, occupational, and speech therapy, skill-building activities, counseling, and brain injury support groups.
- A prolonged period of rehabilitation for those whose coma exceeds 24 hours and have major persistent neurologic sequelae.

The AI generated answer 

# **Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?**

In [ ]:
ground,rel = generate_faithfulness_and_relevance_score(
    user_input=query_5,
    system_prompt=system_message,
    retriever=retriever,
    faithfulness_rater_system_message=faithfulness_rater_system_message,
    relevance_rater_system_message=relevance_rater_system_message
)

print("Faithfulness metric : \n", ground,end="\n\n\n\n")
print("Relevance metric : \n",rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Faithfulness metric : 
  To evaluate the answer, we need to follow these steps:

1. Identify the information in the context that relates to the question. In this case, the context provides details about the necessary precautions and treatment for a person with a fractured leg.
2. Determine if the AI generated answer is derived solely from the identified information in the context.
3. If the answer is not derived solely from the context, evaluate to what extent it is derived from the context.

The AI generated answer adheres to the metric as it includes all the necessary precautions and treatment steps for a person with a fractured leg that are mentioned in the context. Therefore, the answer is derived completely from the context.

Based on the evaluation criteria:
1 - not at all
2 - to a limited extent
3 - to a good extent
4 - mostly
5 - completely

The AI generated answer is rated 5 as it is derived completely from the context.



Relevance metric : 
  To evaluate the context as per t

# **8 - Actionable Insights and Business Recommendations**
Key Business Takeaways from the RAG-Based AI Solution (Using Merck Manuals)

The RAG-based AI prototype built on the Merck Manuals demonstrates strong strategic value for healthcare organizations:

1. Accelerated and More Accurate Diagnostics
Provides clinicians with rapid access to trusted medical knowledge, supporting faster diagnostic decisions and more precise treatment planning—particularly critical in emergency scenarios.

2. Reduced Information Overload
Efficiently retrieves relevant medical content from large knowledge repositories, minimizing time spent searching and enabling focused, context-aware insights.

3. Enhanced Patient Care Quality
Supports evidence-based clinical decision-making, contributing to improved patient outcomes and overall care delivery.

4. Standardization of Clinical Practices
Centralizes access to authoritative medical references, promoting consistency in diagnostic and treatment approaches across departments and providers.

5. Scalable Foundation for Advanced AI Systems
Establishes a strong framework for future AI-driven healthcare assistants capable of integrating structured data, clinical notes, imaging insights, and broader medical knowledge systems.

# **Actionable Insights and Recommendations**

# **Strategic Recommendations for Advancing a Production-Ready RAG-Based Healthcare AI System**

To transition from a prototype to a scalable, production-grade RAG-based AI solution in healthcare, the following strategic actions are recommended:

**1.Integrate More Advanced, Medically Tuned LLMs**

While the current implementation demonstrates feasibility using a lightweight model, production environments require more powerful Large Language Models with strong reasoning and domain-specific fine-tuning. Medically specialized LLMs can significantly enhance diagnostic accuracy, contextual understanding, and response reliability.


**2. Optimize and Fine-Tune Embedding Models**
Embedding quality directly impacts retrieval performance. Healthcare organizations should experiment with embedding models trained on medical corpora to better capture domain-specific terminology and semantic relationships. Fine-tuning embeddings for clinical contexts can substantially improve retrieval precision and contextual alignment.


**3. Refine Chunking Strategies and Retrieval Parameters**
Performance depends heavily on how knowledge is segmented and retrieved. Systematic experimentation with chunk sizes, overlap strategies, and the number of retrieved documents (k) is essential. The goal is to strike an optimal balance between contextual completeness and signal precision, ensuring that the LLM receives the most relevant and coherent information.


**4. Implement Advanced Prompt Engineering Techniques**

Move beyond basic prompting strategies by incorporating advanced techniques such as:

Few-shot prompting

Chain-of-thought reasoning

Tree-of-thought exploration

Chain-of-verification

Response ensembling

These approaches can enhance reasoning depth, reduce hallucinations, and improve synthesis of retrieved medical evidence. Different medical query types may require tailored prompting strategies.


**5. Develop More Sophisticated RAG Architectures**
Consider evolving toward advanced RAG frameworks that include:

Iterative or multi-stage retrieval

Re-ranking mechanisms for improved document prioritization

Hybrid search (vector + keyword)

Integration with knowledge graphs for structured medical reasoning

Such architectures can provide richer and more reliable contextual grounding for clinical decision support.


**6. Establish Continuous Evaluation and Monitoring**
Deploy a robust evaluation framework combining automated techniques (e.g., LLM-as-a-judge) with expert clinical review. Continuous monitoring ensures response faithfulness, medical accuracy, and contextual relevance while enabling systematic performance improvements over time.




